# ASTRA Real-Camera Domain Adaptation (V4-REAL)

This notebook fine-tunes the synthetic ST-GCN V3 model onto real-camera physical RTMPose data collected from a webcam. It performs GPU-accelerated pose extraction, applies exact V3 preprocessing, trains the model conservatively, and evaluates the final checkpoint.

## 1. Setup Environment & Mount Google Drive
Expects Google Drive to contain `real_camera_astra/clips/<action>/*.mp4`

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/real_camera_astra'
assert os.path.exists(DRIVE_DIR), "Please upload the real_camera_astra folder to your Google Drive root."

## 2. Clone ASTRA & Install Dependencies


In [ ]:
# Do NOT reinstall PyTorch. Colab already has the perfect CUDA 12 PyTorch version.
!pip uninstall -y onnxruntime onnxruntime-gpu
# Install ONNX Runtime GPU explicitly for CUDA 12 (Colab default)
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!pip install -q rtmlib opencv-python-headless scikit-learn numpy pandas seaborn matplotlib


## 3. Verify GPU & ONNX Runtime Providers
This is CRITICAL to ensure RTMPose runs at high FPS.

In [ ]:
import torch
import onnxruntime as ort

print("--- GPU VERIFICATION ---")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

print("\n--- ONNX RUNTIME VERIFICATION ---")
providers = ort.get_available_providers()
print(f"Providers: {providers}")
if 'CUDAExecutionProvider' not in providers:
    print("WARNING: CUDAExecutionProvider not found! RTMPose will run on CPU. Please restart runtime or reinstall onnxruntime-gpu.")
else:
    print("CUDAExecutionProvider found. RTMPose will be hardware accelerated.")

## 4. Extract Real RTMPose Data from Clips
We load YOLO and RTMPose, then iterate through the clips in Drive.

In [ ]:
import cv2
import json
import time
import numpy as np
from pathlib import Path
from ml.models.pose.model import RTMPoseEstimator
from ml.models.detection.model import YOLODetector
import torch
import onnxruntime as ort

print("--- TASK 2 & 3: VERIFY DEVICE & ACTUAL PROVIDERS ---")
# 1. Initialize models ONCE
print("Initializing YOLO (cuda)...")
det = YOLODetector(device="cuda")
det.load_model("checkpoints/detection/YOLO11S-ASTRA-v1/weights/best.pt")

print("Initializing RTMPose (cuda, onnxruntime)...")
pose_model = RTMPoseEstimator(device="cuda", backend="onnxruntime")
pose_model.load_model(None)

# Drill down into rtmlib internals to get ACTUAL providers
try:
    pose_session = pose_model._pose_model.pose_model.session
    print(f"RTMPose pose providers: {pose_session.get_providers()}")
    if 'CUDAExecutionProvider' not in pose_session.get_providers():
        print("CRITICAL ERROR: RTMPose internal session is NOT using CUDAExecutionProvider!")
    else:
        print("SUCCESS: RTMPose is using CUDAExecutionProvider.")
except Exception as e:
    print(f"Could not read RTMPose internal session: {e}")

try:
    # If rtmlib has a built-in detector session:
    det_session = pose_model._pose_model.det_model.session
    print(f"RTMPose detector providers: {det_session.get_providers()}")
except:
    pass

try:
    print(f"YOLO device: {det.model.device}")
except:
    pass

print("\n--- TASK 1 & 4 & 5: BENCHMARK EXACTLY ONE CLIP ---")
CLIPS_DIR = Path(DRIVE_DIR) / "clips"
EXTRACT_DIR = Path("data/real_camera/extracted")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

idle_dir = CLIPS_DIR / "idle"
files = list(idle_dir.glob("*.mp4"))

if not files:
    print("No idle clips found!")
else:
    f = files[0] # Pick exactly ONE clip
    print(f"Benchmarking Clip: {f.name}")
    
    out_json = EXTRACT_DIR / f"{f.stem}_poses.json"
    if out_json.exists(): # TASK 9: Cache Check
        print(f"Clip {f.name} already extracted. Skipping...")
    else:
        cap = cv2.VideoCapture(str(f))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"Original frames: {total_frames}")
        
        # Select exactly 60
        if total_frames > 60:
            indices = np.linspace(0, total_frames - 1, 60, dtype=int)
        else:
            indices = np.arange(total_frames)
            
        print(f"Selected frames: {len(indices)}")
        
        pose_sequence = []
        confs = []
        
        yolo_times = []
        pose_times = []
        
        clip_t_start = time.time()
        
        for i, idx in enumerate(indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: continue
            
            # YOLO Time
            t0 = time.time()
            detections = det.detect(frame)
            t1 = time.time()
            yolo_times.append(t1 - t0)
            
            # RTMPose Time
            poses = pose_model.predict(frame, detections)
            t2 = time.time()
            pose_times.append(t2 - t1)
            
            if poses:
                kps = poses[0].keypoints_2d.tolist()
                conf = np.mean([k[2] for k in poses[0].keypoints_2d]) if poses[0].keypoints_2d.shape[1] == 3 else 1.0
                pose_sequence.append(kps)
                confs.append(conf)
            else:
                pose_sequence.append(None)
                confs.append(0.0)
                
            # TASK 8: Progress Logging
            if (i + 1) % 5 == 0 or (i + 1) == len(indices):
                elapsed = time.time() - clip_t_start
                avg_iter = elapsed / (i + 1)
                eta = avg_iter * (len(indices) - (i + 1))
                print(f"Pose extraction: {i+1}/{len(indices)} | Elapsed: {elapsed:.1f}s | ETA: {eta:.1f}s")
                
        cap.release()
        clip_t_total = time.time() - clip_t_start
        
        print("\n--- TASK 6: BENCHMARK RESULTS ---")
        avg_yolo = np.mean(yolo_times) * 1000 if yolo_times else 0
        avg_pose = np.mean(pose_times) * 1000 if pose_times else 0
        print(f"YOLO average: {avg_yolo:.2f} ms/frame")
        print(f"RTMPose average: {avg_pose:.2f} ms/frame")
        print(f"Total clip extraction: {clip_t_total:.2f} seconds")
        print(f"RTMPose calls: {len(pose_times)}")
        
        # Save cache
        session_id = f.stem
        meta_file = f.with_suffix(".json")
        if meta_file.exists():
            with open(meta_file, "r") as mf:
                try: session_id = json.load(mf).get("session_id", f.stem)
                except: pass
                
        with open(out_json, "w") as jf:
            json.dump({"action": "idle", "clip_id": f.stem, "session_id": session_id, "poses": pose_sequence}, jf)
            
        print("\n--- TENSOR SHAPE CHECK ---")
        tensor = np.zeros((2, 60, 17, 1), dtype=np.float32)
        valid = [np.array(p, dtype=np.float32) for p in pose_sequence if p is not None]
        if valid:
            seq = valid[-60:]
            while len(seq) < 60: seq.append(seq[-1].copy())
            print(f"Tensor successfully constructed with shape: {tensor.shape}")
        else:
            print("No valid poses found to construct tensor!")
            
        print("\nSTOPPING AFTER ONE CLIP AS REQUESTED.")


## 5. Build ST-GCN Tensors [C=2, T=60, V=17, M=1] & Split Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
import os

ACTION2ID = {a: i for i, a in enumerate(ACTIONS)}

class RealActionDataset(Dataset):
    def __init__(self, data_list, max_len=60):
        self.data = data_list
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        poses = item["poses"]
        
        tensor = np.zeros((2, self.max_len, 17, 1), dtype=np.float32)
        valid = [np.array(p, dtype=np.float32) for p in poses if p is not None]
        
        if not valid:
            return torch.tensor(tensor), item["label"]
            
        seq = valid[-self.max_len:]
        while len(seq) < self.max_len:
            seq.append(seq[-1].copy())
            
        seq_np = np.array(seq)
        
        # EXACT V3 PREPROCESSING: Root centering
        for i in range(self.max_len):
            kp = seq_np[i]
            pelvis = (kp[11][:2] + kp[12][:2]) / 2.0
            kp[:, :2] = kp[:, :2] - pelvis
            max_val = np.max(np.abs(kp[:, :2])) + 1e-5
            seq_np[i, :, :2] = kp[:, :2] / max_val
            
        tensor[0, :, :, 0] = seq_np[:, :, 0] # X
        tensor[1, :, :, 0] = seq_np[:, :, 1] # Y
        
        return torch.tensor(tensor), item["label"]

data_list = []
for f in EXTRACT_DIR.glob("*_poses.json"):
    with open(f, "r") as jf:
        d = json.load(jf)
        data_list.append({"clip_id": d["clip_id"], "session_id": d.get("session_id", d["clip_id"]), "poses": d["poses"], "label": ACTION2ID[d["action"]]})

# SPLIT BY SESSION (GroupShuffleSplit to ensure no overlap)
import numpy as np
groups = [d["session_id"] for d in data_list]

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss1.split(data_list, groups=groups))
train_data = [data_list[i] for i in train_idx]
temp_data = [data_list[i] for i in temp_idx]
temp_groups = [groups[i] for i in temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_data, groups=temp_groups))
val_data = [temp_data[i] for i in val_idx]
test_data = [temp_data[i] for i in test_idx]

train_groups = set([d["session_id"] for d in train_data])
val_groups = set([d["session_id"] for d in val_data])
test_groups = set([d["session_id"] for d in test_data])

print("\n--- SPLIT VERIFICATION ---")
print(f"Train groups: {list(train_groups)}")
print(f"Validation groups: {list(val_groups)}")
print(f"Test groups: {list(test_groups)}")

overlap_val = train_groups.intersection(val_groups)
overlap_test = train_groups.intersection(test_groups)
overlap_val_test = val_groups.intersection(test_groups)
assert len(overlap_val) == 0, f"Overlap in Train/Val: {overlap_val}"
assert len(overlap_test) == 0, f"Overlap in Train/Test: {overlap_test}"
assert len(overlap_val_test) == 0, f"Overlap in Val/Test: {overlap_val_test}"
print("Zero overlap verified.")


print(f"Splits -> Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

train_loader = DataLoader(RealActionDataset(train_data), batch_size=16, shuffle=True)
val_loader = DataLoader(RealActionDataset(val_data), batch_size=16, shuffle=False)
test_loader = DataLoader(RealActionDataset(test_data), batch_size=16, shuffle=False)


## 6. Fine-Tune ST-GCN V3 (Conservative)

In [ ]:
import torch.nn as nn
import torch.optim as optim
from ml.models.action.stgcn.model import STGCNPP
from sklearn.metrics import accuracy_score

device = torch.device("cuda")
model = STGCNPP(num_classes=6, in_channels=2).to(device)
model.load_state_dict(torch.load("checkpoints/action/STGCNPP-ASTRA-v3/best.pt", map_location=device))

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_val = 0.0
patience = 8
patience_cnt = 0

out_dir = Path("checkpoints/action/STGCNPP-ASTRA-v4-REAL")
out_dir.mkdir(parents=True, exist_ok=True)
best_path = out_dir / "best.pt"

for epoch in range(50):
    model.train()
    for X, y in train_loader:
        optimizer.zero_grad()
        out = model(X.to(device))
        loss = criterion(out, y.to(device))
        loss.backward()
        optimizer.step()
        
    model.eval()
    val_trues, val_preds = [], []
    with torch.no_grad():
        for X, y in val_loader:
            out = model(X.to(device))
            val_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            val_trues.extend(y.numpy())
            
    val_acc = accuracy_score(val_trues, val_preds)
    print(f"Epoch {epoch+1} - Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= patience:
            print("Early stopping!")
            break

## 7. Evaluate on Held-Out Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(best_path))
model.eval()

test_trues, test_preds = [], []
with torch.no_grad():
    for X, y in test_loader:
        out = model(X.to(device))
        test_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        test_trues.extend(y.numpy())

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(test_trues, test_preds, target_names=ACTIONS))

cm = confusion_matrix(test_trues, test_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=ACTIONS, yticklabels=ACTIONS, cmap="Blues")
plt.title("V4-REAL Confusion Matrix")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()

## 8. Export Checkpoint to Google Drive

In [ ]:
import shutil
drive_export_path = Path(DRIVE_DIR) / "STGCNPP-ASTRA-v4-REAL.zip"
shutil.make_archive("/content/STGCNPP-ASTRA-v4-REAL", 'zip', out_dir)
shutil.copy("/content/STGCNPP-ASTRA-v4-REAL.zip", drive_export_path)
print(f"Successfully exported checkpoint to: {drive_export_path}")
print("Download this file and place it in checkpoints/action/STGCNPP-ASTRA-v4-REAL/")